In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",120)

## Project Configuration

In [2]:
PROJECT_NAME = ("iPrint-News-Recommendation-Ranking-System")
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = None
for path in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if path.name == PROJECT_NAME:
        PROJECT_ROOT = path
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root: {PROJECT_NAME} Make sure the notebook is running inside the project directory.")
print("Project Root:", PROJECT_ROOT)
DATA_DIR = (PROJECT_ROOT / "data")
ARTIFACT_DIR = (PROJECT_ROOT / "artifacts")
CANDIDATE_DIR = (ARTIFACT_DIR / "candidates")
LTR_DIR = (ARTIFACT_DIR / "ltr")
DIVERSITY_DIR = (ARTIFACT_DIR / "diversity_novelty")
DIVERSITY_DIR.mkdir(parents=True, exist_ok=True)
print("Directories:")
print("DATA_DIR:", DATA_DIR)
print("CANDIDATE_DIR:", CANDIDATE_DIR)
print("LTR_DIR:", LTR_DIR)
print("DIVERSITY_DIR:", DIVERSITY_DIR)

Project Root: D:\iPrint-News-Recommendation-Ranking-System
Directories:
DATA_DIR: D:\iPrint-News-Recommendation-Ranking-System\data
CANDIDATE_DIR: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates
LTR_DIR: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr
DIVERSITY_DIR: D:\iPrint-News-Recommendation-Ranking-System\artifacts\diversity_novelty


## Locate Dataset Files

In [3]:
def find_file(project_root, filename):
    matches = list(project_root.rglob(filename))
    if not matches:
        return None
    return matches[0]
CONSUMER_PATH = find_file(PROJECT_ROOT,"consumer_transanctions.csv")
CONTENT_PATH = find_file(PROJECT_ROOT,"platform_content.csv")
if CONSUMER_PATH is None:
    raise FileNotFoundError("consumer_transanctions.csv not found.")
if CONTENT_PATH is None:
    raise FileNotFoundError("platform_content.csv not found.")
print("Consumer data:", CONSUMER_PATH)
print("Content data:", CONTENT_PATH)

Consumer data: D:\iPrint-News-Recommendation-Ranking-System\Dataset\consumer_transanctions.csv
Content data: D:\iPrint-News-Recommendation-Ranking-System\Dataset\platform_content.csv


## Load Consumer And Contest Data

In [4]:
consumer = pd.read_csv(CONSUMER_PATH)
content = pd.read_csv(CONTENT_PATH)
print("Consumer shape:", consumer.shape)
print("Content shape:",  content.shape)

Consumer shape: (72312, 8)
Content shape: (3122, 13)


## Time Stamp Preparations

In [5]:
consumer["event_timestamp"] = pd.to_numeric(consumer["event_timestamp"],errors="coerce")
content["event_timestamp"] = pd.to_numeric(content["event_timestamp"],errors="coerce")
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True,errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
print("Consumer timestamp range:")
print(consumer["event_datetime"].min(), "→", consumer["event_datetime"].max())
print("Content timestamp range:")
print(content["event_datetime"].min(), "→", content["event_datetime"].max())

Consumer timestamp range:
2016-03-14 13:54:36+00:00 → 2017-02-28 19:21:51+00:00
Content timestamp range:
2016-03-28 19:19:39+00:00 → 2017-02-28 18:51:11+00:00


## Normalize IDS

In [6]:
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
content["item_id"] = (content["item_id"].astype(str).str.strip())
content["producer_id"] = (content["producer_id"].astype(str).str.strip())
print("IDs normalized")

IDs normalized


## Latest Contest Availability

In [7]:
content_sorted = (content.sort_values(["item_id","event_datetime"]).copy())
latest_content_state = (content_sorted.groupby("item_id",as_index=False).tail(1).copy())
latest_content_state["is_available"] = (latest_content_state["interaction_type"].eq("content_present"))
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])
pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])
print("Available articles:", len(available_items))
print("Pulled-out articles:", len(pulled_items))

Available articles: 2983
Pulled-out articles: 74


## Article Metadata

In [8]:
article_metadata = (content[["item_id","title","text_description","language","item_type","producer_id","item_url"]].drop_duplicates(subset=["item_id"]).copy())
article_metadata["title"] = (article_metadata["title"].fillna("").astype(str).str.strip())
article_metadata["language"] = (article_metadata["language"].fillna("").astype(str).str.lower().str.strip())
article_metadata["item_type"] = (article_metadata["item_type"].fillna("UNKNOWN").astype(str).str.upper().str.strip())
article_metadata["producer_id"] = (article_metadata["producer_id"].fillna("UNKNOWN").astype(str).str.strip())
print("Article metadata:", article_metadata.shape)

Article metadata: (3057, 7)


## Article First Published Time

In [9]:
article_first_present = (content[content["interaction_type"].eq("content_present")].sort_values(["item_id", "event_datetime"]).drop_duplicates(subset=["item_id"],keep="first")[["item_id","event_datetime"]].rename(columns={"event_datetime": "article_published_at"}))
article_metadata = (article_metadata.drop(columns=["article_published_at"],errors="ignore").merge(article_first_present, on="item_id", how="left",validate="one_to_one"))
print("Publication timestamps:", article_metadata["article_published_at"].notna().sum())

Publication timestamps: 3047


## 1. Load LTR Output

In [10]:
ltr_candidates_path = (LTR_DIR / "final_meta.parquet")
ltr_predictions_path = (LTR_DIR  / "ltr_predictions.parquet")
print("Checking LTR artifacts...")
print("final_meta:", ltr_candidates_path)
print("ltr_predictions:", ltr_predictions_path)

Checking LTR artifacts...
final_meta: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet
ltr_predictions: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\ltr_predictions.parquet


In [11]:
LTR_FILENAMES = ["final_meta.parquet","ltr_predictions.parquet","valid_meta.parquet","recommendations.parquet","production_recommendations.parquet"]
search_directories = [PROJECT_ROOT / "artifacts" / "ltr", PROJECT_ROOT / "artifacts" / "candidates", PROJECT_ROOT / "artifacts", PROJECT_ROOT / "Notebook" / "artifacts" / "ltr", PROJECT_ROOT / "Notebook" / "artifacts", PROJECT_ROOT / "Notebook"]

existing_ltr_files = []
for directory in search_directories:
    if not directory.exists():
        continue
    for filename in LTR_FILENAMES:
        path = directory / filename
        if path.exists() and path.is_file():
            if path not in existing_ltr_files:
                existing_ltr_files.append(path)
if not existing_ltr_files:
    print("LTR file not found in standard directories.")
    print("Searching entire project...")
    for filename in LTR_FILENAMES:
        matches = list(PROJECT_ROOT.rglob(filename))
        for path in matches:
            if path.is_file() and path not in existing_ltr_files:
                existing_ltr_files.append(path)
if not existing_ltr_files:
    print("\n No LTR parquet file found.")
    print("\nProject root:")
    print(PROJECT_ROOT)
    print("\nExpected files:")
    for filename in LTR_FILENAMES:
        print(f"  - {filename}")
    raise FileNotFoundError(
        "\n\nNo LTR prediction file exists.\n"
        "Run Learning_to_Rank.ipynb and save the final "
        "LTR predictions to:\n\n"
        f"{PROJECT_ROOT / 'artifacts' / 'ltr' / 'final_meta.parquet'}"
    )
print("LTR files found:")
for i, path in enumerate(existing_ltr_files, start=1):
    print(f"{i}. {path}")

LTR files found:
1. D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet


In [12]:
ltr_path = existing_ltr_files[0]
print("Loading LTR file:")
print(ltr_path)
ltr_candidates = pd.read_parquet(ltr_path)
print("LTR shape:", ltr_candidates.shape)
print("LTR columns:")
print(ltr_candidates.columns.tolist())
display(ltr_candidates.head())

Loading LTR file:
D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet
LTR shape: (514172, 2)
LTR columns:
['consumer_id', 'item_id']


,consumer_id,item_id
0,-1007001694607905623,-1038011342017850
1,-1007001694607905623,2271592336048425450
2,-1007001694607905623,2244894675266276573
3,-1007001694607905623,2072448887839540892
4,-1007001694607905623,1992928170409443117


In [13]:
ltr_candidates_path = (LTR_DIR / "final_meta.parquet")
ltr_predictions_path = (LTR_DIR / "ltr_predictions.parquet")
print("Checking LTR artifacts...")
print("final_meta:", ltr_candidates_path)
print("ltr_predictions:", ltr_predictions_path)

Checking LTR artifacts...
final_meta: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet
ltr_predictions: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\ltr_predictions.parquet


In [14]:
possible_ltr_files = [LTR_DIR / "final_meta.parquet", LTR_DIR / "ltr_predictions.parquet", LTR_DIR / "valid_meta.parquet", LTR_DIR / "recommendations.parquet", LTR_DIR / "production_recommendations.parquet"]
existing_ltr_files = [path  for path in possible_ltr_files if path.exists()]
if not existing_ltr_files:
    print("No standard LTR parquet found.")
    print("Searching entire project")
    search_names = ["final_meta.parquet", "valid_meta.parquet", "ltr_predictions.parquet","recommendations.parquet"]
    for name in search_names:
        matches = list(PROJECT_ROOT.rglob(name))
        if matches:
            existing_ltr_files.extend(matches)
if not existing_ltr_files:
    raise FileNotFoundError("No LTR output file found. Run Learning_to_Rank.ipynb first and save the LTR predictions.")
print("LTR files found:")
for path in existing_ltr_files:
    print(" -", path)

LTR files found:
 - D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet


## 2. Select LTR Data

In [15]:
preferred_names = ["final_meta.parquet", "ltr_predictions.parquet", "valid_meta.parquet", "recommendations.parquet"]
ltr_path = None
for name in preferred_names:
    matches = [path for path in existing_ltr_files if path.name == name]
    if matches:
        ltr_path = matches[0]
        break
if ltr_path is None:
    ltr_path = existing_ltr_files[0]
print("Using LTR file:", ltr_path)
ltr_candidates = pd.read_parquet(ltr_path)
print("LTR data shape:", ltr_candidates.shape)
print("LTR columns:")
for i, col in enumerate(ltr_candidates.columns, 1):
    print(f"{i:03d}. {col}")

Using LTR file: D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet
LTR data shape: (514172, 2)
LTR columns:
001. consumer_id
002. item_id


## 3. Merge Article Metadata

In [16]:
metadata_columns = ["title", "text_description", "language", "item_type","producer_id","item_url","article_published_at"]
ltr_candidates = (ltr_candidates.drop(columns=[col for col in metadata_columns if col in ltr_candidates.columns],errors="ignore"))
ltr_candidates["item_id"] = (ltr_candidates["item_id"].astype(str).str.strip())
ltr_candidates = (ltr_candidates.merge(article_metadata,on="item_id",how="left",validate="many_to_one"))
print("After metadata merge:", ltr_candidates.shape)

After metadata merge: (514172, 9)


## 4. Validate LTR Data

In [17]:
print("LTR dataframe shape:")
print(ltr_candidates.shape)
print("LTR columns:")
for i, col in enumerate(ltr_candidates.columns, start=1):
    print(f"{i:2d}. {col}")
print("First 5 rows:")
display(ltr_candidates.head())
print("Data types:")
display(ltr_candidates.dtypes)

LTR dataframe shape:
(514172, 9)
LTR columns:
 1. consumer_id
 2. item_id
 3. title
 4. text_description
 5. language
 6. item_type
 7. producer_id
 8. item_url
 9. article_published_at
First 5 rows:


,consumer_id,item_id,title,text_description,language,item_type,producer_id,item_url,article_published_at
0,-1007001694607905623,-1038011342017850,Para entender o Dia Internacional contra a Homofobia,O Dia Internacional contra a Homofobia é comemorado em 17 de maio para lembrar a data em que a OMS (Organização Mund...,pt,HTML,6735372008307093370,https://universidadedocotidiano.catracalivre.com.br/para-entender/para-entender-o-dia-internacional-contra-homofobia/,2016-05-17 16:50:00+00:00
1,-1007001694607905623,2271592336048425450,"Google I/O 2016 Preview: Machine Learning, Virtual Reality And Android N - ARC",The democratization of machine learning continues. Google is about to set its developer agenda for the next year. So...,en,HTML,-1443636648652872475,https://arc.applause.com/2016/05/13/google-io-2016-machine-learning-virtual-reality-android-n/,2016-05-13 22:46:21+00:00
2,-1007001694607905623,2244894675266276573,"Microservices Reference Architecture, Part 4 -12‑Factor App",The NGINX Microservices Reference Architecture is under development. It will be made publicly available later this y...,en,HTML,7645894863578715801,https://www.nginx.com/blog/microservices-reference-architecture-part-4-adapting-the-twelve-factor-app/,2016-07-29 17:33:55+00:00
3,-1007001694607905623,2072448887839540892,"Welcome to GoogleBank, Facebook Bank, Amazon Bank, and Apple Bank - Enrique Dans","Welcome to GoogleBank, Facebook Bank, Amazon Bank, and Apple Bank How would you like to bank with Apple, Google, Ama...",en,HTML,-3390049372067052505,https://medium.com/enrique-dans/welcome-to-googlebank-facebook-bank-amazon-bank-and-apple-bank-c9c3955006d4,2016-05-10 22:51:54+00:00
4,-1007001694607905623,1992928170409443117,Google vai reduzir em 50% consumo de memória do Chrome | Google Discovery,O Google anunciou que a versão 55 do Chrome vai incluir um novo motor JavaScript que irá reduz significativamente o ...,pt,HTML,-4028919343899978105,http://googlediscovery.com/2016/10/11/google-vai-reduzir-em-50-consumo-de-memoria-do-chrome/,2016-10-17 16:46:05+00:00


Data types:


consumer_id                          object
item_id                              object
title                                object
text_description                     object
language                             object
item_type                            object
producer_id                          object
item_url                             object
article_published_at    datetime64[ns, UTC]
dtype: object

In [18]:
possible_ltr_files = [LTR_DIR / "final_meta.parquet",LTR_DIR / "ltr_predictions.parquet", LTR_DIR / "valid_meta.parquet", LTR_DIR / "recommendations.parquet", LTR_DIR / "production_recommendations.parquet"]
existing_ltr_files = [path for path in possible_ltr_files if path.exists()]
if not existing_ltr_files:
    print("No standard LTR parquet found.")
    print("Searching entire project...")
    search_names = ["final_meta.parquet", "valid_meta.parquet", "ltr_predictions.parquet", "recommendations.parquet"]
    for name in search_names:
        matches = list(PROJECT_ROOT.rglob(name))
        if matches:
            existing_ltr_files.extend(matches)
if not existing_ltr_files:
    raise FileNotFoundError("No LTR output file found Run Learning_to_Rank.ipynb first and save the LTR predictions.")
print("LTR files found:")
for path in existing_ltr_files:
    print(" -", path)

LTR files found:
 - D:\iPrint-News-Recommendation-Ranking-System\artifacts\ltr\final_meta.parquet


In [19]:
metadata_columns = ["title", "text_description", "language","item_type", "producer_id", "item_url","article_published_at"]
ltr_candidates = (ltr_candidates.drop(columns=[col for col in metadata_columns if col in ltr_candidates.columns],errors="ignore"))
ltr_candidates["item_id"] = (ltr_candidates["item_id"].astype(str).str.strip())
ltr_candidates = (ltr_candidates.merge(article_metadata,on="item_id",how="left",validate="many_to_one"))
print("After metadata merge:", ltr_candidates.shape)

After metadata merge: (514172, 9)


## 5. Filter English Content

In [20]:
before_count = len(ltr_candidates)
ltr_candidates = ltr_candidates[ltr_candidates["language"].astype(str).str.lower().eq("en")].copy()
after_count = len(ltr_candidates)
print("Candidates before English filter:", before_count)
print("Candidates after English filter:", after_count)
print("Removed:", before_count - after_count)

Candidates before English filter: 514172
Candidates after English filter: 372832
Removed: 141340


## 6. Remove Unavailable Articles

In [21]:
before_count = len(ltr_candidates)
ltr_candidates = ltr_candidates[ltr_candidates["item_id"].isin(available_items)].copy()
after_count = len(ltr_candidates)
print("Candidates before availability filter:", before_count)
print("Candidates after availability filter:", after_count)
print("Removed unavailable:", before_count - after_count)

Candidates before availability filter: 372832
Candidates after availability filter: 372832
Removed unavailable: 0


## 7. Load User History

In [22]:
consumer_history = consumer[["consumer_id","item_id","event_datetime","interaction_type","consumer_session_id","country"]].copy()
consumer_history["consumer_id"] = (consumer_history["consumer_id"].astype(str).str.strip())
consumer_history["item_id"] = (consumer_history["item_id"].astype(str).str.strip())
print("Consumer history:", consumer_history.shape)

Consumer history: (72312, 6)


## 8. Bulid Seen Items & Remove Seen Articles

In [23]:
user_seen_items = (consumer_history.groupby("consumer_id")["item_id"].agg(set).to_dict())
print("Users with interaction history:", len(user_seen_items))

Users with interaction history: 1895


In [24]:
def remove_seen_items(candidates, seen_items,):
    keep_mask = []
    for _, row in candidates.iterrows():
        user_id = row["consumer_id"]
        item_id = row["item_id"]
        seen = seen_items.get(user_id, set())
        keep_mask.append(item_id not in seen)
    return candidates[keep_mask].copy()
before_count = len(ltr_candidates)
ltr_candidates = remove_seen_items(ltr_candidates, user_seen_items)
after_count = len(ltr_candidates)
print("Before seen-item filtering:", before_count)
print("After seen-item filtering:", after_count)
print("Seen items removed:", before_count - after_count)

Before seen-item filtering: 372832
After seen-item filtering: 372403
Seen items removed: 429


## 9. Validate Seen Item Constraints

In [25]:
seen_violations = 0
for _, row in ltr_candidates.iterrows():
    user_id = row["consumer_id"]
    item_id = row["item_id"]
    if item_id in user_seen_items.get(user_id,set()):
        seen_violations += 1
print("Seen-item violations:", seen_violations)
assert (seen_violations == 0), "Seen articles remain in candidates."
print(" No seen articles remain.")

Seen-item violations: 0
 No seen articles remain.


## 10. Calculate Freshness

In [26]:
REFERENCE_TIME = (consumer_history["event_datetime"].max())
FRESHNESS_HALF_LIFE_DAYS = 14.0
ltr_candidates["article_age_days"] = ((REFERENCE_TIME - ltr_candidates["article_published_at"]).dt.total_seconds().div(86400.0).clip(lower=0))
ltr_candidates["freshness_score"] = np.power(0.5,ltr_candidates["article_age_days"] / FRESHNESS_HALF_LIFE_DAYS)
print("Reference time:", REFERENCE_TIME)
display(ltr_candidates[["item_id", "article_age_days","freshness_score"]].head())

Reference time: 2017-02-28 19:21:51+00:00


,item_id,article_age_days,freshness_score
1,2271592336048425450,290.857986,5.570961e-07
2,2244894675266276573,214.074954,2.494192e-05
3,2072448887839540892,293.854132,4.802933e-07
5,1990052153136096105,144.196238,7.933630e-04
6,1929674614667189969,274.185220,1.271834e-06


## 11. User Historical Popularity

In [27]:
article_interaction_counts = (consumer_history.groupby("item_id").size().rename("historical_interactions").reset_index())
ltr_candidates = (ltr_candidates.drop(columns=["historical_interactions"],errors="ignore").merge(article_interaction_counts,on="item_id",how="left"))
ltr_candidates["historical_interactions"] = (ltr_candidates["historical_interactions"].fillna(0))
print("Historical popularity statistics:")
display(ltr_candidates["historical_interactions"].describe())

Historical popularity statistics:


count    372403.000000
mean         53.850267
std          57.900525
min           0.000000
25%          14.000000
50%          33.000000
75%          84.000000
max         433.000000
Name: historical_interactions, dtype: float64

## 12. Popularity Based Novelty

In [28]:
ltr_candidates["raw_novelty"] = (1.0 / np.log2(ltr_candidates["historical_interactions"] + 2.0))
novelty_min = (ltr_candidates["raw_novelty"].min())
novelty_max = (ltr_candidates["raw_novelty"].max())
if novelty_max > novelty_min:
    ltr_candidates["novelty_score"] = (ltr_candidates["raw_novelty"] - novelty_min) / (novelty_max - novelty_min)
else:
    ltr_candidates["novelty_score"] = 1.0
print("Novelty score range:")
print(ltr_candidates["novelty_score"].min(), "→",ltr_candidates["novelty_score"].max())

Novelty score range:
0.0 → 1.0


## 13. User Producer Affinity

In [29]:
user_producer_history = (consumer_history.merge(article_metadata[["item_id","producer_id"]],on="item_id",how="left").groupby(["consumer_id","producer_id"]).size().rename("user_producer_interactions").reset_index())
ltr_candidates = (ltr_candidates.drop(columns=["user_producer_interactions"],errors="ignore").merge(user_producer_history,on=["consumer_id","producer_id"],how="left"))
ltr_candidates["user_producer_interactions"] = (ltr_candidates["user_producer_interactions"].fillna(0))
print("User-producer affinity calculated.")

User-producer affinity calculated.


## 14. Producer Diversity Statistics

In [30]:
producer_global_counts = (consumer_history.groupby("item_id").size().rename("item_interactions").reset_index().merge(article_metadata[["item_id","producer_id"]],on="item_id",how="left").groupby("producer_id")["item_interactions"].sum().rename("producer_interactions").reset_index())
ltr_candidates = (ltr_candidates.drop(columns=["producer_interactions"],errors="ignore").merge(producer_global_counts,on="producer_id",how="left"))
ltr_candidates["producer_interactions"] = (ltr_candidates["producer_interactions"].fillna(0))
display(ltr_candidates[["producer_id","producer_interactions"]].drop_duplicates().sort_values("producer_interactions",ascending=False).head(20))

,producer_id,producer_interactions
4,-1032019229384696495,6598
33,3609194402293569455,5116
0,-1443636648652872475,4652
51,6013226412048763966,2154
1,7645894863578715801,2124
5,3891637997717104548,2007
9,-8020832670974472349,1725
229,1895326251577378793,1724
47,-709287718034731589,1498
2,-3390049372067052505,1301


## 15. Define Diversity Configuration

In [31]:
DIVERSITY_CONFIG = {
    # Final number of recommendations
    "top_k": 10,
    # Maximum number of articles from one producer
    "max_per_producer": 2,
    # Maximum articles of same item type
    "max_per_item_type": 4,
    # MMR relevance weight
    "relevance_weight": 0.70,
    # MMR diversity weight
    "diversity_weight": 0.30,
    # Novelty contribution
    "novelty_weight": 0.10,
    # Freshness contribution
    "freshness_weight": 0.10,
    # User producer affinity contribution
    "producer_affinity_weight": 0.05,
    # Number of candidates considered before MMR
    "candidate_pool_size": 100,
    # Number of recommendations
    "final_top_k": 10,
}
DIVERSITY_CONFIG

{'top_k': 10,
 'max_per_producer': 2,
 'max_per_item_type': 4,
 'relevance_weight': 0.7,
 'diversity_weight': 0.3,
 'novelty_weight': 0.1,
 'freshness_weight': 0.1,
 'producer_affinity_weight': 0.05,
 'candidate_pool_size': 100,
 'final_top_k': 10}

## 16. Normalize LTR Scores

In [32]:
# ============================================================
# Diagnose LTR Candidate Columns
# ============================================================

print("LTR candidates shape:", ltr_candidates.shape)

print("\nAvailable columns:")
for i, col in enumerate(ltr_candidates.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFirst rows:")
display(ltr_candidates.head())

LTR candidates shape: (372403, 16)

Available columns:
01. consumer_id
02. item_id
03. title
04. text_description
05. language
06. item_type
07. producer_id
08. item_url
09. article_published_at
10. article_age_days
11. freshness_score
12. historical_interactions
13. raw_novelty
14. novelty_score
15. user_producer_interactions
16. producer_interactions

First rows:


,consumer_id,item_id,title,text_description,language,item_type,producer_id,item_url,article_published_at,article_age_days,freshness_score,historical_interactions,raw_novelty,novelty_score,user_producer_interactions,producer_interactions
0,-1007001694607905623,2271592336048425450,"Google I/O 2016 Preview: Machine Learning, Virtual Reality And Android N - ARC",The democratization of machine learning continues. Google is about to set its developer agenda for the next year. So...,en,HTML,-1443636648652872475,https://arc.applause.com/2016/05/13/google-io-2016-machine-learning-virtual-reality-android-n/,2016-05-13 22:46:21+00:00,290.857986,5.570961e-07,7.0,0.315465,0.227307,0.0,4652
1,-1007001694607905623,2244894675266276573,"Microservices Reference Architecture, Part 4 -12‑Factor App",The NGINX Microservices Reference Architecture is under development. It will be made publicly available later this y...,en,HTML,7645894863578715801,https://www.nginx.com/blog/microservices-reference-architecture-part-4-adapting-the-twelve-factor-app/,2016-07-29 17:33:55+00:00,214.074954,2.494192e-05,24.0,0.212746,0.111359,1.0,2124
2,-1007001694607905623,2072448887839540892,"Welcome to GoogleBank, Facebook Bank, Amazon Bank, and Apple Bank - Enrique Dans","Welcome to GoogleBank, Facebook Bank, Amazon Bank, and Apple Bank How would you like to bank with Apple, Google, Ama...",en,HTML,-3390049372067052505,https://medium.com/enrique-dans/welcome-to-googlebank-facebook-bank-amazon-bank-and-apple-bank-c9c3955006d4,2016-05-10 22:51:54+00:00,293.854132,4.802933e-07,134.0,0.141094,0.030480,0.0,1301
3,-1007001694607905623,1990052153136096105,Introducing Ask a Female Engineer,"We've recruited a group of female engineers with years of industry experience to try an experiment with us called ""A...",en,HTML,-534549863526737439,http://themacro.com/articles/2016/09/introducing-ask-a-female-engineer/,2016-10-07 14:39:16+00:00,144.196238,7.933630e-04,41.0,0.184289,0.079237,1.0,312
4,-1007001694607905623,1929674614667189969,Diane Greene wants to put the enterprise front and center of Google Cloud strategy,"When Google bought bebop Technologies last fall for $348 million , it got more than a stealthy startup. It also land...",en,HTML,-1032019229384696495,http://techcrunch.com/2016/05/30/diane-greene-wants-to-put-the-enterprise-front-and-center-of-google-cloud-strategy/,2016-05-30 14:55:08+00:00,274.185220,1.271834e-06,60.0,0.167949,0.060793,0.0,6598


## 17. Load Semantic Embeddings

In [37]:
EMBEDDING_PATH = (ARTIFACT_DIR / "features" / "article_embeddings.npy")
EMBEDDING_INDEX_PATH = (ARTIFACT_DIR / "features" / "article_embedding_index.parquet")
if (EMBEDDING_PATH.exists() and EMBEDDING_INDEX_PATH.exists()):
    article_embeddings = np.load(EMBEDDING_PATH)
    embedding_metadata = pd.read_parquet(EMBEDDING_INDEX_PATH)
    print("Article embeddings loaded.")
    print("Embedding shape:", article_embeddings.shape)
else:
    article_embeddings = None
    embedding_metadata = None
    print("Article embedding artifacts not found.")
    print("MMR will use title/text similarity fallback.")

Article embeddings loaded.
Embedding shape: (2166, 384)


In [39]:
embedding_lookup = {}
if article_embeddings is not None:
    for _, row in embedding_metadata.iterrows():
        item_id = str(row["item_id"]).strip()
        embedding_index = int(row["embedding_index"])
        embedding_lookup[item_id] = article_embeddings[embedding_index]
print("Embedding lookup size:", len(embedding_lookup))

Embedding lookup size: 2166


## Text Similarity Fallback

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidf_vectorizer = None
tfidf_matrix_diversity = None
tfidf_item_to_index = {}
if article_embeddings is None:
    diversity_catalog = (article_metadata[["item_id","title","text_description"]].drop_duplicates("item_id").copy())
    diversity_catalog["article_text"] = (diversity_catalog["title"].fillna("") + " " + diversity_catalog["text_description"].fillna(""))
    tfidf_vectorizer = (TfidfVectorizer(max_features=10000,stop_words="english",ngram_range=(1, 2),min_df=2))
    tfidf_matrix_diversity = (tfidf_vectorizer.fit_transform(diversity_catalog["article_text"]))
    tfidf_item_to_index = {item_id: idx for idx, item_id in enumerate(diversity_catalog["item_id"])}
    print("TF-IDF fallback matrix:", tfidf_matrix_diversity.shape)
else:
    print("Using semantic embeddings.")

Using semantic embeddings.


In [42]:
# ============================================================
# Item Similarity
# ============================================================

def item_similarity(
    item_a,
    item_b,
):

    if item_a == item_b:
        return 1.0


    # --------------------------------------------------------
    # Semantic embedding similarity
    # --------------------------------------------------------

    if (
        item_a in embedding_lookup
        and item_b in embedding_lookup
    ):

        vector_a = embedding_lookup[
            item_a
        ]

        vector_b = embedding_lookup[
            item_b
        ]

        similarity = float(
            np.dot(
                vector_a,
                vector_b
            )
        )

        return float(
            np.clip(
                similarity,
                -1.0,
                1.0
            )
        )


    # --------------------------------------------------------
    # TF-IDF fallback
    # --------------------------------------------------------

    if (
        item_a in tfidf_item_to_index
        and item_b in tfidf_item_to_index
    ):

        idx_a = tfidf_item_to_index[
            item_a
        ]

        idx_b = tfidf_item_to_index[
            item_b
        ]

        similarity = cosine_similarity(
            tfidf_matrix_diversity[idx_a],
            tfidf_matrix_diversity[idx_b]
        )[0, 0]

        return float(
            np.clip(
                similarity,
                0.0,
                1.0
            )
        )


    return 0.0

In [46]:
def mmr_rerank(candidates, top_k=10, relevance_column="base_relevance_score",item_column="item_id", producer_column="producer_id", item_type_column="item_type",max_per_producer=2, max_per_item_type=4, relevance_weight=0.70, diversity_weight=0.30,):
    if candidates.empty:
        return candidates.copy()
    df = candidates.copy()
    df[relevance_column] = pd.to_numeric(df[relevance_column],errors="coerce").fillna(0.0)
    df = (df.sort_values(relevance_column, ascending=False).drop_duplicates(subset=[item_column]).reset_index(drop=True))
    selected_indices = []
    selected_items = []
    producer_counts = Counter()
    item_type_counts = Counter()
    while (len(selected_indices) < min(top_k, len(df))):
        best_index = None
        best_score = -np.inf
        for idx, row in df.iterrows():
            if idx in selected_indices:
                continue
            item_id = str(row[item_column])
            producer = str(row.get(producer_column,"UNKNOWN"))
            item_type = str(row.get(item_type_column,"UNKNOWN"))
            if (producer_counts[producer] >= max_per_producer):
                continue
            if (item_type_counts[item_type] >= max_per_item_type):
                continue
            relevance = float(row[relevance_column])
            if not selected_items:
                max_similarity = 0.0
            else:
                similarities = [item_similarity(item_id,selected_item) for selected_item in selected_items]
                max_similarity = max(similarities)
            mmr_score = (relevance_weight * relevance - diversity_weight * max_similarity)
            if mmr_score > best_score:
                best_score = (mmr_score)
                best_index = idx
        if best_index is None:
            break
        selected_indices.append(best_index)
        selected_row = df.loc[best_index]
        selected_item = str(selected_row[item_column])
        selected_producer = str(selected_row[producer_column])
        selected_item_type = str(selected_row[item_type_column])
        selected_items.append(selected_item)
        producer_counts[selected_producer] += 1
        item_type_counts[selected_item_type] += 1
    result = (df.loc[selected_indices].copy().reset_index(drop=True))
    result["mmr_rank"] = (np.arange(len(result)) + 1)
    result["mmr_score"] = np.nan
    selected_items = []
    for idx, row in result.iterrows():
        item_id = str(row[item_column])
        relevance = float(row[relevance_column])
        if not selected_items:
            max_similarity = 0.0
        else:
            max_similarity = max(item_similarity(item_id,selected_item)  for selected_item in selected_items)
        result.loc[idx, "mmr_score"] = (relevance_weight * relevance  - diversity_weight * max_similarity)
        selected_items.append(item_id)
    return result